# Chapter 3 - Statistical Experiments and Significance Testing


**Nama:** Aryo Zaki Abdillah  
**NIM:** 101032300060  
**Kelas:** TK47-02  
**Universitas:** Telkom University

> Reproduksi kode dan pendalaman teoretis dari *Practical Statistics for Data Scientists* (O'Reilly), edisi ke-2.
>
> Struktur topik mengikuti notebook Python chapter 1–4 pada repository resmi:
> https://github.com/gedeck/practical-statistics-for-data-scientists

## Tujuan

Chapter ini membahas desain eksperimen statistik dan pengujian signifikansi.
Topik utamanya adalah A/B testing, control group, hypothesis testing,
permutation test, p-value, statistical significance, Type I/II error,
t-test, multiple testing, ANOVA, chi-square test, Fisher's exact test,
multi-arm bandit, serta power dan sample size.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

## 1. A/B Testing

A/B testing membandingkan dua kondisi untuk melihat apakah outcome berbeda.

In [ ]:
group_A = np.array([20, 22, 21, 19, 23, 20, 24, 21, 22, 20])
group_B = np.array([25, 26, 24, 27, 23, 26, 25, 27, 24, 26])

difference = group_B.mean() - group_A.mean()

print("Mean A:", group_A.mean())
print("Mean B:", group_B.mean())
print("Observed difference:", difference)

## 2. Control Group dan Treatment Group

Control group menjadi baseline, sedangkan treatment group menerima perubahan yang sedang diuji.

## 3. Hypothesis Testing

Null hypothesis (H0) biasanya menyatakan tidak ada perbedaan/efek. Alternative hypothesis (H1) menyatakan terdapat perbedaan/efek.

## 4. Permutation Test

Permutation test membentuk distribusi referensi melalui pengacakan ulang label observasi.

In [ ]:
observed = group_B.mean() - group_A.mean()

combined = np.concatenate([group_A, group_B])
permutation_diffs = []

for _ in range(10000):
    shuffled = np.random.permutation(combined)
    perm_A = shuffled[:len(group_A)]
    perm_B = shuffled[len(group_A):]
    permutation_diffs.append(perm_B.mean() - perm_A.mean())

permutation_diffs = np.array(permutation_diffs)

p_perm = np.mean(
    np.abs(permutation_diffs) >= abs(observed)
)

print("Observed difference:", observed)
print("Permutation p-value:", p_perm)

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(permutation_diffs, bins=40)
plt.axvline(observed, linestyle="--")
plt.title("Permutation Distribution")
plt.xlabel("Difference in means")
plt.ylabel("Frequency")
plt.show()

## 5. p-value

p-value menunjukkan seberapa ekstrem hasil observasi jika data dianalisis di bawah asumsi null hypothesis. p-value bukan probabilitas bahwa H0 benar.

## 6. Statistical Significance

In [ ]:
alpha = 0.05

if p_perm < alpha:
    print("Hasil berada di bawah significance level 0.05.")
else:
    print("Hasil tidak berada di bawah significance level 0.05.")

## 7. Type I dan Type II Error

Type I error adalah false positive, yaitu menolak H0 saat H0 benar. Type II error adalah false negative, yaitu tidak menolak H0 saat efek sebenarnya ada.

## 8. t-Test

In [ ]:
t_result = stats.ttest_ind(
    group_A,
    group_B,
    equal_var=False
)

print("t statistic:", t_result.statistic)
print("p-value:", t_result.pvalue)

## 9. Multiple Testing

Ketika banyak hypothesis diuji sekaligus, peluang mendapatkan hasil signifikan secara kebetulan meningkat. Koreksi multiple testing dapat digunakan untuk mengendalikan masalah tersebut.

In [ ]:
from statsmodels.stats.multitest import multipletests

p_values = np.array([0.001, 0.012, 0.021, 0.08, 0.20])

adjusted = multipletests(
    p_values,
    alpha=0.05,
    method="holm"
)

print("Original p-values:", p_values)
print("Adjusted p-values:", adjusted[1])

## 10. ANOVA

ANOVA digunakan untuk menguji perbedaan mean pada lebih dari dua kelompok.

In [ ]:
group_1 = np.array([10, 11, 12, 13, 12])
group_2 = np.array([14, 15, 16, 15, 14])
group_3 = np.array([18, 17, 19, 18, 20])

anova = stats.f_oneway(
    group_1,
    group_2,
    group_3
)

print("F statistic:", anova.statistic)
print("p-value:", anova.pvalue)

## 11. Chi-Square Test

Chi-square test dapat digunakan untuk mengevaluasi hubungan dua variabel kategorikal.

In [ ]:
table = np.array([
    [30, 20],
    [10, 40]
])

chi = stats.chi2_contingency(table)

print("Chi-square:", chi.statistic)
print("p-value:", chi.pvalue)
print("Degrees of freedom:", chi.dof)

## 12. Fisher's Exact Test

Fisher's exact test dapat digunakan untuk tabel kontingensi 2x2 dan berguna ketika jumlah observasi kecil.

In [ ]:
table = np.array([
    [8, 2],
    [1, 9]
])

fisher = stats.fisher_exact(table)

print("Odds ratio:", fisher.statistic)
print("p-value:", fisher.pvalue)

## 13. Multi-Arm Bandit

Multi-arm bandit menggambarkan eksperimen dengan beberapa pilihan. Exploration mencari informasi tentang pilihan lain, sedangkan exploitation menggunakan pilihan yang performanya sudah diketahui.

In [ ]:
np.random.seed(42)

true_rates = np.array([0.10, 0.14, 0.08])
counts = np.zeros(3, dtype=int)
successes = np.zeros(3, dtype=int)

epsilon = 0.1

for _ in range(1000):
    if np.random.rand() < epsilon:
        arm = np.random.randint(0, 3)
    else:
        rates = np.divide(
            successes,
            counts,
            out=np.zeros(3, dtype=float),
            where=counts > 0
        )
        arm = np.argmax(rates)

    reward = np.random.rand() < true_rates[arm]
    counts[arm] += 1
    successes[arm] += int(reward)

print("Pull counts:", counts)
print("Observed success rates:", successes / counts)

## 14. Power dan Sample Size

Statistical power adalah peluang mendeteksi efek ketika efek tersebut memang ada. Power dipengaruhi antara lain oleh effect size, sample size, variability, dan significance level.

In [ ]:
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()

required_n = analysis.solve_power(
    effect_size=0.5,
    alpha=0.05,
    power=0.80,
    ratio=1,
    alternative="two-sided"
)

print("Estimated sample size per group:", np.ceil(required_n))

## Interpretasi

Pengujian statistik membutuhkan desain eksperimen yang jelas dan definisi
hipotesis yang tepat. Permutation test dan t-test dapat digunakan untuk
membandingkan dua kelompok, sedangkan ANOVA digunakan untuk beberapa kelompok.
p-value membantu mengevaluasi hasil terhadap null hypothesis, tetapi tidak
menyatakan probabilitas H0 benar.

## Kesimpulan Chapter 3

Eksperimen statistik menghubungkan data dengan pengambilan keputusan berbasis
hipotesis. Pemilihan metode harus mempertimbangkan tipe data, desain eksperimen,
ukuran sample, dan pertanyaan analisis.